In [1]:
from groq import Groq
from dotenv import load_dotenv
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [2]:
from pydantic import BaseModel

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

In [3]:
CalendarEvent.model_json_schema()

{'properties': {'name': {'title': 'Name', 'type': 'string'},
  'date': {'title': 'Date', 'type': 'string'},
  'participants': {'items': {'type': 'string'},
   'title': 'Participants',
   'type': 'array'}},
 'required': ['name', 'date', 'participants'],
 'title': 'CalendarEvent',
 'type': 'object'}

In [4]:
import json

schema = CalendarEvent.model_json_schema()

response = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": f"Extract the event information. Return JSON only matching this schema: {schema}"},
        {"role": "user", "content": "Alice and Bob are going to a science fair on Friday."},
    ],
    response_format={"type": "json_object"}
)

raw = response.choices[0].message.content
print(raw)
event = CalendarEvent(**json.loads(raw))
print(event)

{
  "name": "Science Fair",
   "date": "Friday",
   "participants": [
      "Alice",
      "Bob"
   ]
}
name='Science Fair' date='Friday' participants=['Alice', 'Bob']


In [5]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

Indexed 385 chunks from 95 documents


In [6]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

In [7]:
import json

instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

In [8]:
def llm(user_prompt, instructions=None, model="llama-3.3-70b-versatile"):

    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })
    
    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = groq_client.chat.completions.create(
        model=model,
        messages=messages
    )

    return (response.choices[0].message.content)

In [9]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt, instructions)

In [10]:
answer = rag('how do I implement LLM as a Judge?')

In [61]:
def llm_structured(user_prompt, output_type, instructions=None, model="llama-3.3-70b-versatile"):
    schema = output_type.model_json_schema()
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions + f"\n\nYour response must be a JSON object with the following structure:\n{schema}\n\nReturn only the JSON object with actual values, not the schema itself."
        })
    
    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = groq_client.chat.completions.create(
        model=model,
        messages=messages,
        response_format={"type": "json_object"}
    )

    raw = response.choices[0].message.content.strip()
    print(raw)
    event = output_type(**json.loads(raw))
    return(event)

In [18]:
response = llm_structured(
    instructions="Extract the event information.",
    user_prompt="Alice and Bob are going to a science fair on Friday.",
    output_type=CalendarEvent
)

In [19]:
response

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

In [20]:
class RAGResponse(BaseModel):
    answer: str
    found_answer: bool

In [21]:
def rag_structured(query, output_type=RAGResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(
        instructions=instructions,
        user_prompt=prompt,
        output_type=output_type,
    )

In [22]:
answer = rag_structured('how do i do llm evals?')

print(answer.answer[:100])
print(answer.found_answer)

To do LLM evals, you can use the CorrectnessLLMEval descriptor, which compares the generated respons
True


In [23]:
RAGResponse.model_json_schema()

{'properties': {'answer': {'title': 'Answer', 'type': 'string'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['answer', 'found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [24]:
from typing import Optional

class RAGResponse(BaseModel):
    answer: Optional[str] = None
    found_answer: bool

In [25]:
RAGResponse.model_json_schema()

{'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Answer'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [26]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


In [27]:
instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.

If you don't find the answer, set `answer` to None
"""

In [28]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


In [29]:
instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

In [36]:
class RAGResponse(BaseModel):
    """
    The response from the documentation RAG system

    If the answer to the question wasn't found in the database, `answer` is None
    """
    answer: Optional[str] = None
    found_answer: bool

In [37]:
RAGResponse.model_json_schema()

{'description': "The response from the documentation RAG system\n\nIf the answer to the question wasn't found in the database, `answer` is None",
 'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Answer'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [38]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


In [40]:
from pydantic import Field

class RAGResponse(BaseModel):
    """
    The response from the documentation RAG system
    """
    answer: Optional[str] = Field(None, description="Answer to the question or None if it's not found")
    found_answer: bool = Field(description="True if the answer is found, False otherwise")

In [41]:
RAGResponse.model_json_schema()

{'description': 'The response from the documentation RAG system',
 'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'description': "Answer to the question or None if it's not found",
   'title': 'Answer'},
  'found_answer': {'description': 'True if the answer is found, False otherwise',
   'title': 'Found Answer',
   'type': 'boolean'}},
 'required': ['found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [46]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


In [47]:
from typing import Literal

class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [48]:
RAGResponse.model_json_schema()

{'description': 'This model provides a structured answer with metadata about the response,\nincluding confidence, categorization, and follow-up suggestions.',
 'properties': {'answer': {'description': "The main answer to the user's question in markdown",
   'title': 'Answer',
   'type': 'string'},
  'found_answer': {'description': 'True if relevant information was found in the documentation',
   'title': 'Found Answer',
   'type': 'boolean'},
  'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
   'title': 'Confidence',
   'type': 'number'},
  'confidence_explanation': {'description': 'Explanation about the confidence level',
   'title': 'Confidence Explanation',
   'type': 'string'},
  'answer_type': {'description': 'The category of the answer',
   'enum': ['how-to',
    'explanation',
    'troubleshooting',
    'comparison',
    'reference'],
   'title': 'Answer Type',
   'type': 'string'},
  'followup_questions': {'description': 'S

In [53]:
answer = rag_structured('how do I evaluate llms', RAGResponse)

In [54]:
print(answer.answer[:100])
print(answer.confidence)
print(answer.confidence_explanation)
print(answer.answer_type)
print(answer.followup_questions)

To evaluate LLMs, you can use multiple LLMs to evaluate the same output and obtain an aggregate eval
0.95
The answer is based on a clear step-by-step guide provided in the context, which explains how to evaluate LLMs using multiple LLMs and Evidently.
how-to
['What are the benefits of using multiple LLMs to evaluate the same output?', 'How do I define a custom descriptor to flag disagreements among LLMs?', 'What are the different types of evaluation prompts that can be used for LLMs?']


In [55]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

In [56]:
print(answer.answer[:100])
print(answer.confidence)
print(answer.confidence_explanation)
print(answer.answer_type)
print(answer.followup_questions)

The provided context does not contain information about installing Kafka on Windows.
1.0
The context does not mention Kafka or Windows, so it is certain that the answer is not found.
reference
['What is the correct documentation for installing Kafka on Windows?', 'Are there any specific requirements for running Kafka on a Windows environment?']


In [57]:
from pydantic import model_validator


class AnswerNotFound(BaseModel):
    explanation: str


class AnswerResponse(BaseModel):
    """
    If answer is found, 'answer' is populated.
    If no answer is found, 'answer_not_found' is populated.
    Only one of the two fields can be set at a time. Never both or neither.
    """

    answer_not_found: Optional[AnswerNotFound] = None
    found_answer: bool
    answer: Optional[RAGResponse] = None

    @model_validator(mode="after")
    def check_consistency(self):
        if self.answer is not None and self.answer_not_found is not None:
            raise ValueError("Provide either 'answer' or 'answer_not_found', not both.")

        if self.answer is None and self.answer_not_found is None:
            raise ValueError("Provide either 'answer' or 'answer_not_found'.")

        return self

In [58]:
AnswerResponse.model_json_schema()

{'$defs': {'AnswerNotFound': {'properties': {'explanation': {'title': 'Explanation',
     'type': 'string'}},
   'required': ['explanation'],
   'title': 'AnswerNotFound',
   'type': 'object'},
  'RAGResponse': {'description': 'This model provides a structured answer with metadata about the response,\nincluding confidence, categorization, and follow-up suggestions.',
   'properties': {'answer': {'description': "The main answer to the user's question in markdown",
     'title': 'Answer',
     'type': 'string'},
    'found_answer': {'description': 'True if relevant information was found in the documentation',
     'title': 'Found Answer',
     'type': 'boolean'},
    'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
     'title': 'Confidence',
     'type': 'number'},
    'confidence_explanation': {'description': 'Explanation about the confidence level',
     'title': 'Confidence Explanation',
     'type': 'string'},
    'answer_type': 

In [62]:
answer = rag_structured('how do I install kafka on windows?', AnswerResponse)
answer

{
  "answer": null,
   "found_answer": false,
   "confidence": 0.0,
   "confidence_explanation": "There is no information about installing Kafka on Windows in the provided context.",
   "answer_type": null,
   "followup_questions": [
      "How do I find information about installing Kafka on Windows?",
      "What are the system requirements for installing Kafka on Windows?"
   ]
}


ValidationError: 1 validation error for AnswerResponse
  Value error, Provide either 'answer' or 'answer_not_found'. [type=value_error, input_value={'answer': None, 'found_a...ing Kafka on Windows?']}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error